# 2026 test set

One-time evaluation on 2026. Every model uses the settings `baselines.ipynb` chose on 2025, trained on 2021–2024. Nothing here is tuned.

In Colab, run this first:
```
!git clone -b test-2026 https://github.com/elizabethahong/ML-Grad-Project.git
%cd ML-Grad-Project/baselines
```

In [1]:
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from data import BVP_FEATURES, FEATURES, load_splits
from eval import TARGETS, compare, evaluate

SEED = 0
splits = load_splits(include_test=True)
train, val, test = splits["train"], splits["val"], splits["test"]
Y_tr = train[TARGETS]
print(len(train), len(val), len(test), test.season.unique())

FEATURE_SETS = {
    "pitcher only": [c for c in FEATURES if not c.startswith("lu_")],
    "lineup only": [c for c in FEATURES if not c.startswith("p_")],
    "combined": FEATURES,
    "combined + bvp": FEATURES + BVP_FEATURES,
}

19436 4860 4858 [2026]


## Frozen settings (copied from the `baselines.ipynb` output)

In [2]:
# ridge alphas as powers of 10 (points on the logspace(-2, 5, 15) grid)
RIDGE = {k: [10.0 ** e for e in v] for k, v in {
    "pitcher only": [3, 1.5, -2, 3.5],
    "lineup only": [3.5, 3, 4, 3.5],
    "combined": [3, 3, 1.5, 3.5],
    "combined + bvp": [3, 3, 2.5, 3.5],
}.items()}
# (max_depth, min_samples_leaf, max_features) per target
RF = {
    "pitcher only": [(8, 5, 0.33), (12, 50, 0.33), (12, 50, 0.2), (12, 20, 0.2)],
    "lineup only": [(8, 20, 0.2), (8, 5, 0.33), (12, 100, 0.2), (12, 100, 0.2)],
    "combined": [(12, 20, 0.2), (12, 5, 0.33), (12, 100, 0.33), (12, 50, 0.2)],
    "combined + bvp": [(12, 20, 0.2), (12, 5, 0.33), (12, 100, 0.33), (12, 50, 0.2)],
}
# (num_leaves, learning_rate, trees) per target
LGB = {
    "pitcher only": [(7, 0.02, 365), (7, 0.02, 268), (4, 0.02, 339), (15, 0.02, 320)],
    "lineup only": [(15, 0.02, 166), (4, 0.02, 172), (4, 0.02, 212), (7, 0.05, 49)],
    "combined": [(4, 0.02, 662), (15, 0.05, 152), (15, 0.05, 54), (15, 0.05, 123)],
    "combined + bvp": [(15, 0.05, 68), (7, 0.02, 389), (15, 0.02, 186), (15, 0.02, 208)],
}


def fit_predict(model_name, set_name, frames):
    cols = FEATURE_SETS[set_name]
    if model_name == "ridge":
        m = make_pipeline(StandardScaler(), Ridge(alpha=RIDGE[set_name])).fit(train[cols], Y_tr)
        return [m.predict(f[cols]) for f in frames]
    out = [[] for _ in frames]
    for t, p in zip(TARGETS, (RF if model_name == "random forest" else LGB)[set_name]):
        if model_name == "random forest":
            m = RandomForestRegressor(n_estimators=300, max_depth=p[0], min_samples_leaf=p[1],
                                      max_features=p[2], n_jobs=-1, random_state=SEED)
        else:
            m = lgb.LGBMRegressor(n_estimators=p[2], num_leaves=p[0], learning_rate=p[1],
                                  min_child_samples=50, subsample=0.8, subsample_freq=1,
                                  colsample_bytree=0.8, random_state=SEED, verbose=-1)
        m.fit(train[cols], Y_tr[t])
        for i, f in enumerate(frames):
            out[i].append(m.predict(f[cols]))
    return [np.column_stack(o) for o in out]

In [3]:
# takes a few minutes
MODELS = ["ridge", "random forest", "lightgbm"]
val_preds, test_preds = {}, {}
for set_name in FEATURE_SETS:
    for model_name in MODELS:
        val_preds[(model_name, set_name)], test_preds[(model_name, set_name)] = fit_predict(model_name, set_name, [val, test])

## Check: frozen settings reproduce the saved 2025 numbers

In [4]:
def rmse_table(preds, frame):
    t = pd.DataFrame({k: evaluate(frame[TARGETS], p)["RMSE"] for k, p in preds.items()}).T
    t.index.names = ["model", "features"]
    return t.round(3)


saved = pd.read_csv("ablation_2025.csv").set_index(["model", "features"])
redone = rmse_table(val_preds, val).reindex(saved.index)
print("max difference vs ablation_2025.csv:", float((redone - saved).abs().max().max()))

max difference vs ablation_2025.csv: 0.0


## Results on 2026

In [5]:
results = {
    "train mean": evaluate(test[TARGETS], np.tile(Y_tr.mean().values, (len(test), 1))),
    "season avg": evaluate(test[TARGETS], test[["p_std_h", "p_std_k", "p_std_bb", "p_std_er"]].values),
}
for m in MODELS:
    results[m] = evaluate(test[TARGETS], test_preds[(m, "combined")])
table = compare(results).round(3)
table.to_csv("results_2026.csv")
table

target          hits        strikeouts         walks        earned_runs       
                 MAE   RMSE        MAE   RMSE    MAE   RMSE         MAE   RMSE
model                                                                         
train mean     1.836  2.291      2.013  2.493  1.059  1.313       1.627  1.991
season avg     1.826  2.306      1.875  2.352  1.068  1.363       1.655  2.111
ridge          1.707  2.124      1.754  2.196  1.001  1.262       1.573  1.948
random forest  1.700  2.120      1.735  2.179  1.005  1.264       1.573  1.950
lightgbm       1.700  2.119      1.731  2.178  1.004  1.263       1.575  1.955

In [6]:
abl = rmse_table(test_preds, test)
abl = abl.reindex(pd.MultiIndex.from_product([MODELS, FEATURE_SETS], names=abl.index.names))
abl.to_csv("ablation_2026.csv")
abl

target                         hits  strikeouts  walks  earned_runs
model         features                                             
ridge         pitcher only    2.129       2.229  1.268        1.947
              lineup only     2.289       2.452  1.305        1.992
              combined        2.124       2.196  1.262        1.948
              combined + bvp  2.124       2.195  1.262        1.948
random forest pitcher only    2.127       2.214  1.268        1.950
              lineup only     2.287       2.455  1.307        1.992
              combined        2.120       2.179  1.264        1.950
              combined + bvp  2.117       2.179  1.263        1.948
lightgbm      pitcher only    2.124       2.212  1.266        1.952
              lineup only     2.289       2.454  1.305        1.990
              combined        2.119       2.178  1.263        1.955
              combined + bvp  2.119       2.175  1.262        1.951

## Bootstrap on 2026 (same method as on 2025)

In [7]:
def rmse(y, p):
    return np.sqrt(np.mean((np.asarray(y) - np.asarray(p)) ** 2))


def pct_gain(y, base, new):
    return 100 * (1 - rmse(y, new) / rmse(y, base))


rng = np.random.default_rng(SEED)
groups = list(test.groupby("game_pk").indices.values())
boot = [np.concatenate([groups[i] for i in rng.integers(len(groups), size=len(groups))])
        for _ in range(2000)]

rows = []
for base_set, new_set in [("pitcher only", "combined"), ("combined", "combined + bvp")]:
    for m in MODELS:
        a, b = test_preds[(m, base_set)], test_preds[(m, new_set)]
        for j, t in enumerate(TARGETS):
            y = test[t].values
            g = np.array([pct_gain(y[i], a[i, j], b[i, j]) for i in boot])
            rows.append({"comparison": f"{base_set} -> {new_set}", "model": m, "target": t,
                         "gain %": pct_gain(y, a[:, j], b[:, j]),
                         "ci low": np.percentile(g, 2.5), "ci high": np.percentile(g, 97.5),
                         "share <= 0": (g <= 0).mean()})
boot_table = pd.DataFrame(rows).set_index(["comparison", "model", "target"]).round(3)
boot_table.to_csv("bootstrap_2026.csv")
boot_table

gain %  ci low  ci high  \
comparison                 model         target                                 
pitcher only -> combined   ridge         hits          0.218  -0.007    0.434   
                                         strikeouts    1.501   0.968    2.006   
                                         walks         0.461   0.122    0.826   
                                         earned_runs  -0.064  -0.215    0.087   
                           random forest hits          0.294   0.102    0.467   
                                         strikeouts    1.584   1.062    2.074   
                                         walks         0.319   0.026    0.631   
                                         earned_runs  -0.000  -0.180    0.170   
                           lightgbm      hits          0.205  -0.030    0.441   
                                         strikeouts    1.570   0.955    2.135   
                                         walks         0.281  -0.024    0.608   
                                         earned_runs  -0.158  -0.382    0.073   
combined -> combined + bvp ridge         hits          0.014  -0.029    0.057   
                                         strikeouts    0.023  -0.039    0.086   
                                         walks        -0.014  -0.087    0.059   
                                         earned_runs   0.010  -0.006    0.027   
                           random forest hits          0.179  -0.011    0.370   
                                         strikeouts    0.014  -0.157    0.185   
                                         walks         0.059  -0.034    0.147   
                                         earned_runs   0.071  -0.007    0.153   
                           lightgbm      hits         -0.010  -0.199    0.197   
                                         strikeouts    0.107  -0.103    0.315   
                                         walks         0.036  -0.118    0.189   
                                         earned_runs   0.226   0.078    0.379   

                                                      share <= 0  
comparison                 model         target                   
pitcher only -> combined   ridge         hits              0.028  
                                         strikeouts        0.000  
                                         walks             0.005  
                                         earned_runs       0.816  
                           random forest hits              0.000  
                                         strikeouts        0.000  
                                         walks             0.018  
                                         earned_runs       0.510  
                           lightgbm      hits              0.040  
                                         strikeouts        0.000  
                                         walks             0.039  
                                         earned_runs       0.909  
combined -> combined + bvp ridge         hits              0.264  
                                         strikeouts        0.248  
                                         walks             0.642  
                                         earned_runs       0.117  
                           random forest hits              0.032  
                                         strikeouts        0.447  
                                         walks             0.102  
                                         earned_runs       0.042  
                           lightgbm      hits              0.542  
                                         strikeouts        0.152  
                                         walks             0.328  
                                         earned_runs       0.001